# Лабораторна робота № 3
## Діагностика шкал вимірювання

Курс СК26 «Інтелектуальний аналіз та візуалізація даних»

**Тривалість:** 2 год аудиторно + 3 год самостійно · **Оцінка:** 4 бали
**Файли:** `Ковід_дані_навчальні.xlsx`, `ЛР3_помилковий_аналіз.ipynb`

---

Шкала вимірювання визначає, які операції над змінною взагалі мають сенс.
Комп'ютер цього не знає: він порахує середнє від номера телефону й не поскаржиться.
Знати має вимірювач.

У цій роботі ви складаєте **паспорт восьми змінних** свого варіанта, показуєте на
власних даних, які статистики для них недопустимі, і виправляєте чужий аналіз,
де таких помилок десять.

In [1]:
ПРІЗВИЩЕ_ІМЯ = 'Недосіка Назарій'      # напишіть тут своє прізвище та ім'я
ГРУПА        = 'ШІ-32'
НОМЕР        = 14      # ваш номер за списком групи (1–30)

assert ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР, 'Заповніть три поля вище'
assert 1 <= НОМЕР <= 30, 'Номер має бути від 1 до 30'

In [2]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_colwidth', 60)

ВАРІАНТИ = json.load(open('lr3_variants_student.json', encoding='utf-8'))
В = ВАРІАНТИ[НОМЕР - 1]

print('=' * 74)
print(f'ВАРІАНТ {В["варіант"]}   ·   {ПРІЗВИЩЕ_ІМЯ}   ·   {ГРУПА}')
print('=' * 74)
print()
print('Ваші вісім стовпців:')
for i, c in enumerate(В['стовпці'], 1):
    print(f'   {i}. {c}')
print()
print('Ваша пара для аналізу зв’язку:', ' та '.join(В['пара']))
print()
print('Ваше додаткове питання:')
print('  ', В['питання'])

ВАРІАНТ 14   ·   Недосіка Назарій   ·   ШІ-32

Ваші вісім стовпців:
   1. номер і/х
   2. КТ опис
   3. ЛН
   4. Виписка
   5. КТ
   6. Прокальцитонін
   7. БРзаг2
   8. Сечов1

Ваша пара для аналізу зв’язку: ступінь КТ та ЛН

Ваше додаткове питання:
   У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.


In [3]:
ФАЙЛ = 'Ковід_дані_навчальні.xlsx'
df = pd.read_excel(ФАЙЛ, sheet_name='Лист1', engine='openpyxl')
МОЇ = В['стовпці']

print('Розмір набору:', df.shape)
df[МОЇ].head(8)

Розмір набору: (121, 77)


,номер і/х,КТ опис,ЛН,Виписка,КТ,Прокальцитонін,БРзаг2,Сечов1
0,2377/239,CO-Rads5,ІІІ-IV,2022-03-10 00:00:00,20-30%,0.126,17.0,5.1
1,1121/19,CO-Rads6,ІІ-ІІІ,2022-03-16 00:00:00,30-35%,NaN,13.6,6.6
2,283/674,"КТ-4, CO-Rads6",ІІ,2022-01-18 00:00:00,70-80%,NaN,18.7,7.0
3,2372/513,NaN,ІІІ-IV,2022-01-13 00:00:00,0.8,NaN,13.6,9.5
4,1441/671,NaN,ІІ-ІІІ,2022-03-11 00:00:00,NaN,NaN,15.3,5.2
5,2573/354,"КТ-2, СО-Rads6",ІІ-ІІІ,2021-12-26 00:00:00,30-40%,NaN,15.3,14.0
6,1762/555,"КТ-3, СО-Rads5",ІІ-ІІІ,2022-05-11 00:00:00,60-65%,NaN,15.3,5.7
7,2664/348,"КТ-1, СО-Rads5",І,2022-05-28 00:00:00,до 17%,NaN,15.3,9.1


---
# Етап 1. Паспорт змінних

Для кожного з восьми стовпців визначте шкалу вимірювання. Спирайтеся не на тип даних
у pandas, а на **зміст величини**: що означають її значення і які порівняння між ними
мають сенс.

Нагадування:

| Шкала | Ознака |
|---|---|
| Номінальна | значення можна лише порівняти на рівність |
| Порядкова | є природний порядок, але відстані між сусідніми рівнями невідомі |
| Інтервальна | різниці мають сенс, нуль умовний |
| Відношень | є природний нуль, відношення мають сенс |
| Абсолютна | природні і нуль, і одиниця вимірювання |

Окремо позначайте **ідентифікатори** — значення, що не є величиною взагалі
(номер історії хвороби, прізвище, телефон).

In [4]:
# Спершу подивіться, що каже pandas — і не довіряйте йому
df[МОЇ].info()

<class 'pandas.DataFrame'>
RangeIndex: 121 entries, 0 to 120
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   номер і/х       121 non-null    object 
 1   КТ опис         10 non-null     str    
 2   ЛН              121 non-null    str    
 3   Виписка         120 non-null    object 
 4   КТ              44 non-null     object 
 5   Прокальцитонін  46 non-null     float64
 6   БРзаг2          101 non-null    float64
 7   Сечов1          117 non-null    float64
dtypes: float64(3), object(3), str(2)
memory usage: 8.9+ KB


In [5]:
# Далі — що є всередині насправді
for c in МОЇ:
    з = df[c].dropna()
    типи = ', '.join(f'{t.__name__}×{n}' for t, n in з.map(type).value_counts().items())
    print(f'--- {c}')
    print(f'    непорожніх {len(з)} з {len(df)} | унікальних {з.nunique()} | типи: {типи}')
    print(f'    приклади: {[repr(v) for v in з.unique()[:4]]}')
    print()

--- номер і/х
    непорожніх 121 з 121 | унікальних 119 | типи: int×65, str×56
    приклади: ["'2377/239'", "'1121/19'", "'283/674'", "'2372/513'"]

--- КТ опис
    непорожніх 10 з 121 | унікальних 10 | типи: str×10
    приклади: ["'CO-Rads5'", "'CO-Rads6'", "'КТ-4, CO-Rads6'", "'КТ-2, СО-Rads6'"]

--- ЛН
    непорожніх 121 з 121 | унікальних 6 | типи: str×121
    приклади: ["'ІІІ-IV'", "'ІІ-ІІІ'", "'ІІ'", "'І'"]

--- Виписка
    непорожніх 120 з 121 | унікальних 90 | типи: datetime×118, str×2
    приклади: ['datetime.datetime(2022, 3, 10, 0, 0)', 'datetime.datetime(2022, 3, 16, 0, 0)', 'datetime.datetime(2022, 1, 18, 0, 0)', 'datetime.datetime(2022, 1, 13, 0, 0)']

--- КТ
    непорожніх 44 з 121 | унікальних 26 | типи: float×27, str×17
    приклади: ["'20-30%'", "'30-35%'", "'70-80%'", '0.8']

--- Прокальцитонін
    непорожніх 46 з 121 | унікальних 17 | типи: float×46
    приклади: ['np.float64(0.126)', 'np.float64(0.9)', 'np.float64(0.1)', 'np.float64(0.104)']

--- БРзаг2
    непорож

### Заповніть паспорт

Для кожного стовпця вкажіть шкалу та **обґрунтування** — одне речення про те,
чому саме ця шкала. Обґрунтування оцінюється, назва шкали сама по собі — ні.

In [6]:
ПАСПОРТ = {
    'номер і/х': (
        'ідентифікатор',
        'Номер історії хвороби використовується для позначення запису пацієнта, а не для вимірювання величини.'
    ),
    'КТ опис': (
        'номінальна',
        'Початкові записи є складеними текстовими описами без єдиного порядку; компоненти КТ та CO-RADS потрібно виділити й класифікувати окремо.'
    ),
    'ЛН': (
        'порядкова',
        'Ступені легеневої недостатності мають природний порядок, але різниця між сусідніми ступенями невідома.'
    ),
    'Виписка': (
        'інтервальна',
        'Дата виписки дозволяє обчислювати різницю в днях, але не має природного нуля.'
    ),
    'КТ': (
        'абсолютна',
        'Відсоток ураження легень має природний нуль і фіксовану одиницю вимірювання.'
    ),
    'Прокальцитонін': (
        'відношень',
        'Концентрація прокальцитоніну має природний нуль, тому відношення значень мають сенс.'
    ),
    'БРзаг2': (
        'відношень',
        'Кількісний лабораторний показник має природний нуль, тому можна порівнювати значення за відношенням.'
    ),
    'Сечов1': (
        'відношень',
        'Концентрація сечовини має природний нуль, тому можна визначати, у скільки разів одне значення більше за інше.'
    )
}

for c in МОЇ:
    ПАСПОРТ.setdefault(c, ('', ''))

паспорт_df = pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПАСПОРТ[c][1]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Обґрунтування']
)

паспорт_df

,Стовпець,Шкала,Обґрунтування
0,номер і/х,ідентифікатор,Номер історії хвороби використовується для позначення за...
1,КТ опис,номінальна,Початкові записи є складеними текстовими описами без єди...
2,ЛН,порядкова,"Ступені легеневої недостатності мають природний порядок,..."
3,Виписка,інтервальна,"Дата виписки дозволяє обчислювати різницю в днях, але не..."
4,КТ,абсолютна,Відсоток ураження легень має природний нуль і фіксовану ...
5,Прокальцитонін,відношень,"Концентрація прокальцитоніну має природний нуль, тому ві..."
6,БРзаг2,відношень,"Кількісний лабораторний показник має природний нуль, том..."
7,Сечов1,відношень,"Концентрація сечовини має природний нуль, тому можна виз..."


In [7]:
# --- САМОПЕРЕВІРКА 1 ---
ДОПУСТИМІ = {'номінальна', 'порядкова', 'інтервальна', 'відношень',
             'абсолютна', 'ідентифікатор'}
for c in МОЇ:
    ш, об = ПАСПОРТ[c]
    assert ш.strip().lower() in ДОПУСТИМІ, f'{c}: шкалу «{ш}» не розпізнано'
    assert len(об.strip()) >= 25, f'{c}: обґрунтування закоротке'
print('Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.')

Самоперевірка 1 пройдена: усі вісім стовпців мають шкалу та обґрунтування.


---
# Етап 2. Допустимі статистики

Для кожного стовпця обчисліть **тільки те, що дозволяє його шкала**.

* номінальна та ідентифікатор — мода, частоти
* порядкова — додатково медіана та квантилі
* інтервальна — додатково середнє та стандартне відхилення
* відношень і абсолютна — усе, зокрема коефіцієнт варіації

Якщо для стовпця не можна порахувати нічого, крім частот, — так і напишіть.

In [8]:
def кт_у_частку(значення):
    import re

    if pd.isna(значення):
        return np.nan

    if isinstance(значення, (int, float, np.number)):
        число = float(значення)
        частка = число if 0 <= число <= 1 else число / 100
        return частка if 0 <= частка <= 1 else np.nan

    текст = str(значення).strip().lower().replace(',', '.')
    текст = текст.replace('–', '-').replace('—', '-')
    # Для "до 17%" відома лише верхня межа, тому точкове значення не вигадуємо.
    if текст.startswith('до') or '<' in текст or '≤' in текст:
        return np.nan

    діапазон = re.fullmatch(r'(\d+(?:\.\d+)?)\s*%?\s*-\s*(\d+(?:\.\d+)?)\s*%?', текст)
    if діапазон:
        нижня, верхня = map(float, діапазон.groups())
        if нижня > верхня:
            return np.nan
        # Текстові діапазони трактуємо як відсотки; середина є наближенням.
        if not (0 <= нижня <= верхня <= 100):
            return np.nan
        return (нижня + верхня) / 2 / 100

    if re.fullmatch(r'\d+(?:\.\d+)?\s*%?', текст):
        число = float(текст.replace('%', '').strip())
        частка = число / 100 if '%' in текст or число > 1 else число
        return частка if 0 <= частка <= 1 else np.nan

    return np.nan


def допустима_статистика(стовпець, шкала):
    """Повертає словник статистик, дозволених для цієї шкали."""
    з = df[стовпець].dropna()
    р = {'непорожніх': len(з), 'унікальних': з.nunique()}
    ш = шкала.strip().lower()

    if ш == 'ідентифікатор':
        р['частоти'] = з.value_counts().head(5).to_dict()
        return р

    р['мода'] = з.mode().iloc[0] if not з.empty else None
    р['частоти'] = з.value_counts().head(5).to_dict()

    if ш == 'порядкова':
        if стовпець == 'ЛН':
            порядок = ['І', 'І-ІІ', 'ІІ', 'ІІ-ІІІ', 'ІІІ', 'ІІІ-IV', 'IV']
            ранги = з.map({v: i for i, v in enumerate(порядок)}).dropna().sort_values()
            р['у розрахунку'] = len(ранги)

            if not ранги.empty:
                # Квантиль - перша категорія, накопичена частка якої досягає q.
                квантилі = {
                    q: порядок[int(ранги.iloc[int(np.ceil(q * len(ранги))) - 1])]
                    for q in (0.25, 0.5, 0.75)
                }
                р['медіана'] = квантилі[0.5]
                р['квантилі'] = квантилі

        return р

    if ш in ('інтервальна', 'відношень', 'абсолютна'):
        if стовпець == 'Виписка':
            числа = pd.to_datetime(з, errors='coerce', format='mixed', dayfirst=True)
        elif стовпець == 'КТ':
            числа = з.map(кт_у_частку)
            р['правило обробки КТ'] = (
                'Усі значення подано частками від 0 до 1. '
                'Діапазони замінено їхніми серединами як наближенням. '
                'Записи з верхньою межею (наприклад, до 17%) та нерозпізнані '
                'значення виключено з точкових статистик.'
            )
            р['виключені записи КТ'] = з[числа.isna()].value_counts().to_dict()
        else:
            числа = pd.to_numeric(з, errors='coerce')

        числа = числа.dropna()
        р['у розрахунку'] = len(числа)

        if not числа.empty:
            if стовпець == 'КТ':
                р['мода'] = числа.mode().iloc[0]
                р['частоти'] = числа.value_counts().head(5).to_dict()
            р['медіана'] = числа.median()
            р['квантилі'] = числа.quantile([0.25, 0.5, 0.75]).to_dict()
            р['середнє'] = числа.mean()
            р['стандартне відхилення'] = числа.std()

            if ш in ('відношень', 'абсолютна'):
                if pd.api.types.is_numeric_dtype(числа):
                    середнє = числа.mean()
                    if середнє != 0:
                        р['коефіцієнт варіації'] = числа.std() / середнє * 100

    return р


for c in МОЇ:
    print(f'--- {c} ({ПАСПОРТ[c][0]})')
    print('   ', допустима_статистика(c, ПАСПОРТ[c][0]))
    print()


--- номер і/х (ідентифікатор)
    {'непорожніх': 121, 'унікальних': 119, 'частоти': {1837: 2, 2526: 2, '2377/239': 1, '1121/19': 1, '283/674': 1}}

--- КТ опис (номінальна)
    {'непорожніх': 10, 'унікальних': 10, 'мода': 'CO-Rads5', 'частоти': {'CO-Rads5': 1, 'CO-Rads6': 1, 'КТ-4, CO-Rads6': 1, 'КТ-2, СО-Rads6': 1, 'КТ-3, СО-Rads5': 1}}

--- ЛН (порядкова)
    {'непорожніх': 121, 'унікальних': 6, 'мода': 'ІІ-ІІІ', 'частоти': {'ІІ-ІІІ': 59, 'ІІ': 21, 'ІІІ': 17, 'І': 12, 'І-ІІ': 9}, 'у розрахунку': 121, 'медіана': 'ІІ-ІІІ', 'квантилі': {0.25: 'ІІ', 0.5: 'ІІ-ІІІ', 0.75: 'ІІ-ІІІ'}}

--- Виписка (інтервальна)
    {'непорожніх': 120, 'унікальних': 90, 'мода': datetime.datetime(2021, 9, 7, 0, 0), 'частоти': {datetime.datetime(2021, 11, 19, 0, 0): 3, datetime.datetime(2021, 9, 7, 0, 0): 3, datetime.datetime(2021, 9, 15, 0, 0): 3, datetime.datetime(2021, 9, 11, 0, 0): 3, datetime.datetime(2022, 4, 6, 0, 0): 3}, 'у розрахунку': 118, 'медіана': Timestamp('2022-03-15 00:00:00'), 'квантилі': {0.25

---
# Етап 3. Три безглузді числа

`df.describe()` порахує статистики для всього, що схоже на число. Частина цих чисел
не має жодного змісту.

Знайдіть **щонайменше три** такі числа у своєму наборі: обчисліть їх, а потім
поясніть, чому кожне з них безглузде.

In [9]:
df[МОЇ].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
номер і/х,121.0,119.0,1837.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
КТ опис,10,10,CO-Rads5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ЛН,121,6,ІІ-ІІІ,59,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Виписка,120,90,2021-11-19 00:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
КТ,44.0,26.0,0.3,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Прокальцитонін,46.0,NaN,NaN,NaN,0.752957,3.010375,0.043,0.1,0.1,0.1205,18.22
БРзаг2,101.0,NaN,NaN,NaN,16.216832,5.537094,7.4,13.6,15.3,18.7,60.5
Сечов1,117.0,NaN,NaN,NaN,8.93188,8.174368,3.1,5.5,7.2,9.5,68.0


In [10]:
номери = pd.to_numeric(df['номер і/х'], errors='coerce').dropna()
середнє_номерів = номери.mean()

коди_лн = df['ЛН'].dropna().astype('category').cat.codes + 1
середнє_лн = коди_лн.mean()

дати = pd.to_datetime(df['Виписка'], errors='coerce').dropna()
дні = (дати - pd.Timestamp('1970-01-01')).dt.total_seconds() / 86400
cv_дат = дні.std() / дні.mean() * 100

БЕЗГЛУЗДІ = [
    ('номер і/х', 'Середнє арифметичне', round(середнє_номерів, 2),
     'Номери історій хвороби є ідентифікаторами, тому їх середнє нічого не означає.'),

    ('ЛН', 'Середнє кодів категорій', round(середнє_лн, 2),
     'Ступені ЛН мають порядок, але відстані між ними невідомі. Середнє залежить від обраного кодування.'),

    ('Виписка', 'Коефіцієнт варіації (%)', round(cv_дат, 3),
     'Дата має умовний нуль. Коефіцієнт варіації залежить від початку відліку часу.')
]

assert len(БЕЗГЛУЗДІ) >= 3, 'Потрібно щонайменше три приклади'

pd.DataFrame(
    БЕЗГЛУЗДІ,
    columns=['Стовпець', 'Статистика', 'Значення', 'Чому безглузде']
)

,Стовпець,Статистика,Значення,Чому безглузде
0,номер і/х,Середнє арифметичне,1770.120,"Номери історій хвороби є ідентифікаторами, тому їх серед..."
1,ЛН,Середнє кодів категорій,3.570,"Ступені ЛН мають порядок, але відстані між ними невідомі..."
2,Виписка,Коефіцієнт варіації (%),0.576,Дата має умовний нуль. Коефіцієнт варіації залежить від ...


---
# Етап 4. Міра зв'язку для вашої пари

Для двох змінних вашого варіанта треба оцінити зв'язок. Міру обирає **пара шкал**,
а не зручність:

| Поєднання шкал | Міра |
|---|---|
| відношень × відношень | кореляція Пірсона |
| порядкова × порядкова | рангова кореляція Спірмена |
| номінальна × номінальна | χ² та коефіцієнт Крамера (V) |
| відношень × бінарна номінальна | точковий бісеріальний коефіцієнт |
| номінальна × порядкова | коефіцієнт Крамера (V) |

Обґрунтуйте вибір, обчисліть міру й поясніть результат. Окремо покажіть, **що
станеться**, якщо застосувати Пірсона там, де він недопустимий, — і чому отримане
число не можна інтерпретувати.

In [11]:
A, B = В['пара']
print('Ваша пара:', A, '×', B)
print()
print(f'{A}: шкала —', ПАСПОРТ.get(A, ('не у вашому списку',))[0] or '(визначте самостійно)')
print(f'{B}: шкала —', ПАСПОРТ.get(B, ('не у вашому списку',))[0] or '(визначте самостійно)')
print()
print(df[[A, B]].head(6).to_string())

Ваша пара: ступінь КТ × ЛН

ступінь КТ: шкала — не у вашому списку
ЛН: шкала — порядкова

  ступінь КТ      ЛН
0    сердній  ІІІ-IV
1   середній  ІІ-ІІІ
2  критичний      ІІ
3  критичний  ІІІ-IV
4        NaN  ІІ-ІІІ
5   середній  ІІ-ІІІ


In [12]:
ОБРАНА_МІРА = 'Кореляція Спірмена'

ЧОМУ_САМЕ_ВОНА = (
    'Обидві змінні мають порядкову шкалу. '
    'Кореляція Спірмена дозволяє оцінити зв’язок між '
    'їхніми рангами, не вимагаючи рівних відстаней між категоріями.'
)

assert ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40, 'Заповніть обидва поля'

print('Ступінь КТ:')
print(df['ступінь КТ'].value_counts(dropna=False))

print('\nЛН:')
print(df['ЛН'].value_counts(dropna=False))

пара = df[['ступінь КТ', 'ЛН']].copy()

кт = пара['ступінь КТ'].astype('string').str.lower().str.strip()

кт = кт.replace({
    'легка': 'легкий',
    'легкій': 'легкий',
    'середнй': 'середній',
    'сердній': 'середній',
    'серед ступ': 'середній',
    'серед ступеня': 'середній'
})

порядок_кт = {
    'легкий': 1,
    'середній': 2,
    'важкий': 3,
    'вкрай важкий': 4,
    'критичний': 5
}

пара['КТ_ранг'] = кт.map(порядок_кт)

лн = пара['ЛН'].astype('string').str.upper().str.strip()
лн = лн.str.replace('І', 'I', regex=False)
лн = лн.str.replace(' ', '', regex=False)

порядок_лн = {
    'I': 1,
    'I-II': 2,
    'II': 3,
    'II-III': 4,
    'III': 5,
    'III-IV': 6
}

пара['ЛН_ранг'] = лн.map(порядок_лн)

print('\nНевідомі категорії КТ:',
      кт[кт.notna() & пара['КТ_ранг'].isna()].unique())

print('Невідомі категорії ЛН:',
      лн[лн.notna() & пара['ЛН_ранг'].isna()].unique())

дані = пара[['КТ_ранг', 'ЛН_ранг']].dropna().astype(float)

спірмен = дані['КТ_ранг'].corr(
    дані['ЛН_ранг'],
    method='spearman'
)

print('\nКількість пар:', len(дані))
print('Кореляція Спірмена:', round(спірмен, 4))

Ступінь КТ:
ступінь КТ
NaN              79
середній         10
важкий            9
серед ступ        6
легкий            5
критичний         3
вкрай важкий      3
сердній           1
легка             1
легкий            1
вкрай важкий      1
важкий            1
серед ступеня     1
Name: count, dtype: int64

ЛН:
ЛН
ІІ-ІІІ    59
ІІ        21
ІІІ       17
І         12
І-ІІ       9
ІІІ-IV     3
Name: count, dtype: int64

Невідомі категорії КТ: <ArrowStringArray>
[]
Length: 0, dtype: string
Невідомі категорії ЛН: <ArrowStringArray>
[]
Length: 0, dtype: string

Кількість пар: 42
Кореляція Спірмена: 0.6418


In [13]:
пірсон = дані['КТ_ранг'].corr(дані['ЛН_ранг'], method='pearson')

змінені = дані.copy()
змінені['КТ_ранг'] = змінені['КТ_ранг'] ** 2

пірсон_змінений = змінені['КТ_ранг'].corr(
    змінені['ЛН_ранг'], method='pearson'
)

спірмен_змінений = змінені['КТ_ранг'].corr(
    змінені['ЛН_ранг'], method='spearman'
)

print('Кореляція Спірмена:', round(спірмен, 4))
print('Кореляція Пірсона:', round(пірсон, 4))
print('Пірсон після зміни кодів:', round(пірсон_змінений, 4))
print('Спірмен після зміни кодів:', round(спірмен_змінений, 4))

print(
    '\nПіднесення кодів до квадрата зберігає порядок категорій, '
    'але змінює відстані між ними. Тому кореляція Спірмена залишається '
    '0.6418, а кореляція Пірсона змінюється з 0.5996 до 0.5471. '
    'Відстані між порядковими категоріями не задані змістом шкали, '
    'тому результат Пірсона залежить від довільно обраного кодування '
    'і не має однозначної інтерпретації як зв’язок між цими ступенями.'
)


Кореляція Спірмена: 0.6418
Кореляція Пірсона: 0.5996
Пірсон після зміни кодів: 0.5471
Спірмен після зміни кодів: 0.6418

Піднесення кодів до квадрата зберігає порядок категорій, але змінює відстані між ними. Тому кореляція Спірмена залишається 0.6418, а кореляція Пірсона змінюється з 0.5996 до 0.5471. Відстані між порядковими категоріями не задані змістом шкали, тому результат Пірсона залежить від довільно обраного кодування і не має однозначної інтерпретації як зв’язок між цими ступенями.


---
# Етап 5. Підготовка змінних до моделі

Шкала визначає не лише статистику, а й спосіб кодування.

| Шкала | Як готують до моделі |
|---|---|
| Номінальна | one-hot кодування; порядок категорій не вводять |
| Порядкова | впорядкована категорія або цілі коди із **заданим** порядком |
| Інтервальна | стандартизація; для дат — перехід до тривалостей |
| Відношень | нормалізація або стандартизація; можливе логарифмування |
| Ідентифікатор | у модель не беруть узагалі |

Для кожного свого стовпця вкажіть спосіб підготовки — і **виконайте його**
щонайменше для трьох стовпців різних шкал.

In [14]:
ПІДГОТОВКА = {
    'номер і/х': 'Виключити з моделі, оскільки це ідентифікатор пацієнта, а не ознака.',

    'КТ опис': 'Розділити категорії КТ та CO-RADS, очистити записи й закодувати кожну шкалу окремо зі збереженням порядку.',

    'ЛН': 'Перетворити ступені легеневої недостатності на числові ранги із заданим порядком.',

    'Виписка': 'Перетворити дати на кількість днів від базової дати, за потреби виконати стандартизацію.',

    'КТ': 'Привести всі значення до єдиного формату часток від 0 до 1, обробити діапазони та пропуски.',

    'Прокальцитонін': 'Перетворити на числовий тип, обробити пропуски та виконати стандартизацію.',

    'БРзаг2': 'Перетворити на числовий тип і виконати Min-Max нормалізацію з перевіркою варіативності.',

    'Сечов1': 'Перетворити на числовий тип, обробити пропуски та виконати стандартизацію.'
}

for c in МОЇ:
    ПІДГОТОВКА.setdefault(c, '')

pd.DataFrame(
    [(c, ПАСПОРТ[c][0], ПІДГОТОВКА[c]) for c in МОЇ],
    columns=['Стовпець', 'Шкала', 'Як готуємо']
)

,Стовпець,Шкала,Як готуємо
0,номер і/х,ідентифікатор,"Виключити з моделі, оскільки це ідентифікатор пацієнта, ..."
1,КТ опис,номінальна,"Розділити категорії КТ та CO-RADS, очистити записи й зак..."
2,ЛН,порядкова,Перетворити ступені легеневої недостатності на числові р...
3,Виписка,інтервальна,"Перетворити дати на кількість днів від базової дати, за ..."
4,КТ,абсолютна,Привести всі значення до єдиного формату часток від 0 до...
5,Прокальцитонін,відношень,"Перетворити на числовий тип, обробити пропуски та викона..."
6,БРзаг2,відношень,Перетворити на числовий тип і виконати Min-Max нормаліза...
7,Сечов1,відношень,"Перетворити на числовий тип, обробити пропуски та викона..."


In [15]:
підготовлені = pd.DataFrame(index=df.index)

порядок_лн = {
    'I': 1,
    'I-II': 2,
    'II': 3,
    'II-III': 4,
    'III': 5,
    'III-IV': 6
}

лн = df['ЛН'].astype('string').str.upper().str.strip()
лн = лн.str.replace('І', 'I', regex=False)

підготовлені['ЛН_код'] = лн.map(порядок_лн)

дати = pd.to_datetime(
    df['Виписка'],
    errors='coerce',
    format='mixed',
    dayfirst=True
)

початкова_дата = дати.min()
підготовлені['Виписка_дні'] = (дати - початкова_дата).dt.days

бр = pd.to_numeric(df['БРзаг2'], errors='coerce')

мінімум = бр.min()
максимум = бр.max()

if бр.notna().sum() == 0:
    raise ValueError('Стовпець БРзаг2 не містить числових значень')

if максимум == мінімум:
    підготовлені['БРзаг2_норм'] = 0.0
    print('БРзаг2 має сталі значення, ознаку варто виключити з моделі')
else:
    підготовлені['БРзаг2_норм'] = (бр - мінімум) / (максимум - мінімум)

print('Перші 10 рядків:')
display(підготовлені.head(10))

print('\nКількість непорожніх значень:')
print(підготовлені.count())

print('\nМінімум і максимум нормалізованого БРзаг2:')
print(підготовлені['БРзаг2_норм'].min())
print(підготовлені['БРзаг2_норм'].max())

Перші 10 рядків:


,ЛН_код,Виписка_дні,БРзаг2_норм
0,6,317.0,0.180791
1,4,323.0,0.116761
2,3,266.0,0.212806
3,6,261.0,0.116761
4,4,318.0,0.148776
5,4,243.0,0.148776
6,4,379.0,0.148776
7,1,396.0,0.148776
8,3,381.0,0.148776
9,5,206.0,NaN



Кількість непорожніх значень:
ЛН_код         121
Виписка_дні    118
БРзаг2_норм    101
dtype: int64

Мінімум і максимум нормалізованого БРзаг2:
0.0
1.0


**Перед нормалізацією** обов'язково перевірте варіативність. Стовпець, у якому
всі значення однакові, дасть нуль у знаменнику — і мовчазний `NaN` замість помилки.

In [16]:
for c in МОЇ:
    з = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(з) and з.nunique() == 1:
        print(f'УВАГА: {c} — сталий стовпець, нормалізація дасть ділення на нуль')
    elif len(з) and з.max() == з.min():
        print(f'УВАГА: {c} — max дорівнює min')
print('Перевірку варіативності виконано.')

Перевірку варіативності виконано.


In [17]:
# --- САМОПЕРЕВІРКА 2 ---
assert all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ), \
    'Для кожного стовпця вкажіть спосіб підготовки'
assert len(БЕЗГЛУЗДІ) >= 3, 'Етап 3 не завершено'
print('Самоперевірка 2 пройдена.')

Самоперевірка 2 пройдена.


---
# Етап 6. Виправлення чужого аналізу

Відкрийте файл **`ЛР3_помилковий_аналіз.ipynb`**. Це звіт, який виконується
без жодної помилки й містить десять хибних місць.

Для кожного з десяти заповніть рядок нижче:

* **розділ** — номер розділу звіту;
* **що не так** — яка шкала порушена або яке припущення хибне;
* **чому число неправильне** — що саме воно означає насправді;
* **як правильно** — коректний варіант або обґрунтована відмова від показника.

In [18]:
ВИПРАВЛЕННЯ = [
    (
        '1',
        'Обчислено середнє арифметичне для результату лікування, який має номінальну шкалу.',
        'Значення 1 і 2 є кодами категорій, тому середнє 1.5 не означає стан пацієнта між життям і смертю.',
        'Потрібно окремо обчислити кількість і частку померлих та тих, хто вижив.'
    ),
    (
        '2',
        'Однакові категорії статі записані по-різному і помилково вважаються чотирма статями.',
        'Позначення ж і жін означають жінок, а ч і чол означають чоловіків. Насправді категорій лише дві.',
        'Обєднати однакові категорії та повторно обчислити частоти і відсотки.'
    ),
    (
        '3',
        'Порядкові категорії тяжкості КТ відсортовано за алфавітом і обчислено середнє їхніх кодів.',
        'Алфавітний порядок не відповідає тяжкості стану, а відстані між ступенями невідомі. Середнє не має змісту.',
        'Виправити назви категорій, вручну задати порядок тяжкості та обчислити частоти, моду або медіану.'
    ),
    (
        '4',
        'Відсотки ураження легень обчислено без попереднього приведення всіх значень до спільного формату.',
        'Стовпець містить числа та текстові діапазони відсотків. Частина спостережень відкидається, тому середнє 39.6% не описує всі наявні дані.',
        'Перетворити відсотки та частки до одного формату, окремо обробити діапазони й повторно обчислити статистики.'
    ),
    (
        '5',
        'Сатурацію проаналізовано без перевірки одиниць вимірювання та некоректно записаних значень.',
        'Мінімум 0.9 при інших значеннях близько 90 вказує на можливе змішування часток і відсотків. Середнє може бути спотвореним.',
        'Перевірити підозрілі значення, привести сатурацію до відсотків та повторно обчислити статистики.'
    ),
    (
        '6',
        'Для дат госпіталізації застосовано коефіцієнт варіації, який не підходить для інтервальної шкали.',
        'Дати не мають природного нуля. Значення 0.595% залежить від обраної початкової дати відліку часу.',
        'Відмовитися від коефіцієнта варіації та використовувати розмах дат або стандартне відхилення у днях.'
    ),
    (
        '7',
        'Звязок між двома номінальними змінними оцінено через кореляцію Пірсона та зроблено висновок про відсутність звязку.',
        'Числові коди позначають категорії. Для двох бінарних змінних Пірсон еквівалентний коефіцієнту фі, але значення 0.177 саме по собі не доводить відсутності статистичного звязку.',
        'Використати таблицю спряженості, критерій хі-квадрат і коефіцієнт Крамера з перевіркою статистичної значущості.'
    ),
    (
        '8',
        'Нормалізацію застосовано до всіх числових стовпців без перевірки шкал та варіативності.',
        'Числові коди не обовязково є кількісними ознаками. Для сталого стовпця ФЕРИТ2 знаменник дорівнює нулю, тому результат містить NaN.',
        'Спочатку відібрати допустимі числові ознаки та перевірити, що максимум більший за мінімум.'
    ),
    (
        '9',
        'Усі рядки з будь-якими пропущеними значеннями видалено методом df.dropna().',
        'Після видалення залишилося нуль пацієнтів. Порожній набір не придатний для навчання моделі.',
        'Обробляти пропуски лише у потрібних ознаках, застосовувати заповнення медіаною або інші відповідні методи.'
    ),
    (
        '10',
        'Дату розтину використано як ознаку для прогнозування летальності.',
        'Дата розтину стає відомою після смерті пацієнта. Її використання створює витік цільової інформації, тому кореляція 1.0 не доводить прогностичної цінності.',
        'Виключити дату розтину та інші ознаки, недоступні на момент прогнозування, і повторно оцінити важливість показників.'
    )
]

assert len(ВИПРАВЛЕННЯ) == 10, f'Знайдено {len(ВИПРАВЛЕННЯ)} помилок із 10'

for р in ВИПРАВЛЕННЯ:
    assert len(р) == 4 and all(str(x).strip() for x in р), 'Заповніть усі чотири поля'
    assert len(str(р[2])) >= 30 and len(str(р[3])) >= 25, 'Пояснення закороткі'

pd.DataFrame(
    ВИПРАВЛЕННЯ,
    columns=['Розділ', 'Що не так', 'Чому число неправильне', 'Як правильно']
)

,Розділ,Що не так,Чому число неправильне,Як правильно
0,1,"Обчислено середнє арифметичне для результату лікування, ...","Значення 1 і 2 є кодами категорій, тому середнє 1.5 не о...",Потрібно окремо обчислити кількість і частку померлих та...
1,2,Однакові категорії статі записані по-різному і помилково...,"Позначення ж і жін означають жінок, а ч і чол означають ...",Обєднати однакові категорії та повторно обчислити частот...
2,3,Порядкові категорії тяжкості КТ відсортовано за алфавіто...,"Алфавітний порядок не відповідає тяжкості стану, а відст...","Виправити назви категорій, вручну задати порядок тяжкост..."
3,4,Відсотки ураження легень обчислено без попереднього прив...,Стовпець містить числа та текстові діапазони відсотків. ...,"Перетворити відсотки та частки до одного формату, окремо..."
4,5,Сатурацію проаналізовано без перевірки одиниць вимірюван...,Мінімум 0.9 при інших значеннях близько 90 вказує на мож...,"Перевірити підозрілі значення, привести сатурацію до від..."
5,6,"Для дат госпіталізації застосовано коефіцієнт варіації, ...",Дати не мають природного нуля. Значення 0.595% залежить ...,Відмовитися від коефіцієнта варіації та використовувати ...
6,7,Звязок між двома номінальними змінними оцінено через кор...,Числові коди позначають категорії. Для двох бінарних змі...,"Використати таблицю спряженості, критерій хі-квадрат і к..."
7,8,Нормалізацію застосовано до всіх числових стовпців без п...,Числові коди не обовязково є кількісними ознаками. Для с...,Спочатку відібрати допустимі числові ознаки та перевірит...
8,9,Усі рядки з будь-якими пропущеними значеннями видалено м...,Після видалення залишилося нуль пацієнтів. Порожній набі...,"Обробляти пропуски лише у потрібних ознаках, застосовува..."
9,10,Дату розтину використано як ознаку для прогнозування лет...,Дата розтину стає відомою після смерті пацієнта. Її вико...,"Виключити дату розтину та інші ознаки, недоступні на мом..."


### Виправте щонайменше три помилки в коді

Оберіть три з десяти та покажіть правильне обчислення поруч із неправильним.

In [19]:
print('=== Розділ 1. Результат лікування ===')

результат = pd.to_numeric(df['помер/жив'], errors='coerce')
правильні_коди = результат[результат.isin([1, 2])]

print('Неправильно: середнє =', round(результат.mean(), 2))
print('Правильно: кількість померлих =', (правильні_коди == 1).sum())
print('Правильно: кількість тих, хто вижив =', (правильні_коди == 2).sum())
print('Летальність (%) =', round((правильні_коди == 1).mean() * 100, 2))


print('\n=== Розділ 6. Дати госпіталізації ===')

дати = pd.to_datetime(
    df['Поступлення'],
    errors='coerce',
    format='mixed',
    dayfirst=True
).dropna()

дні_1970 = (дати - pd.Timestamp('1970-01-01')).dt.total_seconds() / 86400
cv_неправильний = дні_1970.std() / дні_1970.mean() * 100

print('Неправильно: коефіцієнт варіації =', round(cv_неправильний, 3), '%')
print('Правильно: найраніша дата =', дати.min().date())
print('Правильно: найпізніша дата =', дати.max().date())
print('Розмах (днів) =', (дати.max() - дати.min()).days)
print('Стандартне відхилення (днів) =',
      round(дати.std() / pd.Timedelta(days=1), 2))


print('\n=== Розділ 9. Видалення пропусків ===')

неправильно = df.dropna()
правильно = df.dropna(subset=['ЛН', 'БРзаг2'])

print('Початкова кількість пацієнтів:', len(df))
print('Неправильно: після df.dropna() =', len(неправильно))
print('Правильно: після очищення потрібних стовпців =', len(правильно))

=== Розділ 1. Результат лікування ===
Неправильно: середнє = 1.5
Правильно: кількість померлих = 61
Правильно: кількість тих, хто вижив = 60
Летальність (%) = 50.41

=== Розділ 6. Дати госпіталізації ===
Неправильно: коефіцієнт варіації = 0.595 %
Правильно: найраніша дата = 2021-02-25
Правильно: найпізніша дата = 2022-06-03
Розмах (днів) = 463
Стандартне відхилення (днів) = 113.05

=== Розділ 9. Видалення пропусків ===
Початкова кількість пацієнтів: 121
Неправильно: після df.dropna() = 0
Правильно: після очищення потрібних стовпців = 101


---
# Етап 7. Висновки та індивідуальне питання

In [20]:
ВИСНОВКИ = [
    'У наборі даних є 121 запис пацієнтів, але лише 119 унікальних номерів історій хвороби. Це означає, що деякі ідентифікатори повторюються, тому перед аналізом потрібно перевірити можливі дублікати.',

    'Найпоширенішим ступенем легеневої недостатності є ІІ-ІІІ, який зустрічається у 59 зі 121 запису. Оскільки ця змінна має порядкову шкалу, для неї доцільно використовувати частоти, моду та медіану, а не середнє арифметичне.',

    'Між ступенем тяжкості КТ та легеневою недостатністю виявлено позитивний зв’язок. Кореляція Спірмена становить 0.6418 для 42 пацієнтів. Це свідчить про тенденцію до зростання ступеня легеневої недостатності зі збільшенням тяжкості змін на КТ.'
]

assert len(ВИСНОВКИ) >= 3 and all(len(в) >= 60 for в in ВИСНОВКИ), \
    'Потрібні три змістовні висновки'

for i, в in enumerate(ВИСНОВКИ, 1):
    print(f'{i}. {в}')

1. У наборі даних є 121 запис пацієнтів, але лише 119 унікальних номерів історій хвороби. Це означає, що деякі ідентифікатори повторюються, тому перед аналізом потрібно перевірити можливі дублікати.
2. Найпоширенішим ступенем легеневої недостатності є ІІ-ІІІ, який зустрічається у 59 зі 121 запису. Оскільки ця змінна має порядкову шкалу, для неї доцільно використовувати частоти, моду та медіану, а не середнє арифметичне.
3. Між ступенем тяжкості КТ та легеневою недостатністю виявлено позитивний зв’язок. Кореляція Спірмена становить 0.6418 для 42 пацієнтів. Це свідчить про тенденцію до зростання ступеня легеневої недостатності зі збільшенням тяжкості змін на КТ.


In [21]:
print('ВАШЕ ПИТАННЯ:')
print(В['питання'])
print()

ВІДПОВІДЬ = (
    'Основна різниця між інтервальною шкалою та шкалою відношень полягає '
    'в наявності природного нуля. В інтервальній шкалі нуль умовний, '
    'тому можна обчислювати різницю між значеннями, але не їх відношення. '
    'Прикладом у нашому наборі є стовпець Виписка, який містить дати. '
    'Можна визначити, що між 10 і 16 березня пройшло 6 днів, '
    'але не можна стверджувати, що одна дата вдвічі більша за іншу. '
    'Шкала відношень має природний нуль. Прикладом є стовпець Прокальцитонін, '
    'який показує концентрацію речовини в крові. '
    'Для нього можна не лише обчислювати різницю, а й порівнювати значення '
    'за відношенням. Наприклад, концентрація 0.9 у 9 разів більша за 0.1. '
    'Отже, операція ділення одного значення на інше з інтерпретацією '
    'результату як кратності допустима для шкали відношень, '
    'але не для інтервальної шкали.'
)

assert len(ВІДПОВІДЬ.strip()) >= 200, 'Відповідь має бути розгорнутою (від 200 символів)'

import textwrap
print(textwrap.fill(ВІДПОВІДЬ, width=100))

ВАШЕ ПИТАННЯ:
У чому різниця між шкалою інтервалів і шкалою відношень? Знайдіть у файлі по одному прикладу кожної та поясніть, яка операція допустима для другої, але не для першої.

Основна різниця між інтервальною шкалою та шкалою відношень полягає в наявності природного нуля. В
інтервальній шкалі нуль умовний, тому можна обчислювати різницю між значеннями, але не їх
відношення. Прикладом у нашому наборі є стовпець Виписка, який містить дати. Можна визначити, що між
10 і 16 березня пройшло 6 днів, але не можна стверджувати, що одна дата вдвічі більша за іншу. Шкала
відношень має природний нуль. Прикладом є стовпець Прокальцитонін, який показує концентрацію
речовини в крові. Для нього можна не лише обчислювати різницю, а й порівнювати значення за
відношенням. Наприклад, концентрація 0.9 у 9 разів більша за 0.1. Отже, операція ділення одного
значення на інше з інтерпретацією результату як кратності допустима для шкали відношень, але не для
інтервальної шкали.


---
# Етап 8. Декларація про використання інструментів ШІ

Користуватися мовними моделями дозволено. Це нормальна інженерна практика.

**Обов'язковою є прозорість:** ви вказуєте, де саме і як скористалися інструментом —
так само, як у науковій статті вказують використане програмне забезпечення.
Задекларована допомога порушенням не є.

Порушенням є інше: здати роботу, яку ви не можете пояснити. На захисті вас
попросять показати конкретний рядок свого коду, пояснити, чому він саме такий,
і внести в нього невелику зміну на місці.

Для кожного етапу оберіть одне з формулювань:

* `'ні'` — інструментами ШІ не користувався
* `'синтаксис'` — питав про назву функції або синтаксис, код писав сам
* `'код, перевірено'` — код згенеровано, я його прочитав, перевірив і розумію
* `'текст, перероблено'` — згенеровано чернетку тексту, яку я переписав по суті
* `'текст, як є'` — згенерований текст залишено майже без змін

In [22]:
ДЕКЛАРАЦІЯ = {
    'етап 1 — паспорт змінних': 'текст, перероблено',
    'етап 2 — допустимі статистики': 'код, перевірено',
    'етап 3 — безглузді числа': 'код, перевірено',
    'етап 4 — міра зв’язку': 'код, перевірено',
    'етап 5 — підготовка до моделі': 'код, перевірено',
    'етап 6 — виправлення аналізу': 'текст, перероблено',
    'етап 7 — висновки й відповідь': 'текст, перероблено'
}

ЩО_САМЕ_Я_ПЕРЕВІРИВ = (
    'Перевірив роботу коду на кожному етапі лабораторної. '
    'Проаналізував шкали вимірювання, обчислені статистики '
    'та коректність застосування методів аналізу. '
    'Перевірив кореляцію Спірмена, підготовку даних до моделі '
    'та результати виправлення помилкового аналізу. '
    'Текстові пояснення та висновки переробив самостійно.'
)

ПІДТВЕРДЖУЮ = True

In [23]:
# --- ФІНАЛЬНА САМОПЕРЕВІРКА ---
ФОРМУЛЮВАННЯ = {'ні', 'синтаксис', 'код, перевірено', 'текст, перероблено', 'текст, як є'}

перевірки = {
    'заповнено титульні поля':        bool(ПРІЗВИЩЕ_ІМЯ and ГРУПА and НОМЕР),
    'етап 1: паспорт із 8 стовпців':  all(ПАСПОРТ[c][0] and ПАСПОРТ[c][1] for c in МОЇ),
    'етап 3: три безглузді числа':    len(БЕЗГЛУЗДІ) >= 3,
    'етап 4: міру обрано й пояснено': bool(ОБРАНА_МІРА and len(ЧОМУ_САМЕ_ВОНА) >= 40),
    'етап 5: підготовку описано':     all(len(ПІДГОТОВКА[c].strip()) >= 15 for c in МОЇ),
    'етап 6: знайдено 10 помилок':    len(ВИПРАВЛЕННЯ) == 10,
    'етап 7: три висновки':           len(ВИСНОВКИ) >= 3,
    'етап 7: відповідь на питання':   len(ВІДПОВІДЬ.strip()) >= 200,
    'декларацію заповнено':           all(v in ФОРМУЛЮВАННЯ for v in ДЕКЛАРАЦІЯ.values()),
    'опис перевірки ≥ 120 символів':  len(ЩО_САМЕ_Я_ПЕРЕВІРИВ.strip()) >= 120,
    'підтвердження авторства':        ПІДТВЕРДЖУЮ is True,
}

print('ЧЕК-ЛИСТ ГОТОВНОСТІ')
print('=' * 52)
for назва, ок in перевірки.items():
    print(('  ГОТОВО     ' if ок else '  НЕ ГОТОВО ') + назва)
print('=' * 52)

assert all(перевірки.values()), 'Робота не готова до здачі — див. чек-лист вище'
print()
print(f'Робота варіанта {В["варіант"]} готова до захисту.')
print(f'{ПРІЗВИЩЕ_ІМЯ}, {ГРУПА}')

ЧЕК-ЛИСТ ГОТОВНОСТІ
  ГОТОВО     заповнено титульні поля
  ГОТОВО     етап 1: паспорт із 8 стовпців
  ГОТОВО     етап 3: три безглузді числа
  ГОТОВО     етап 4: міру обрано й пояснено
  ГОТОВО     етап 5: підготовку описано
  ГОТОВО     етап 6: знайдено 10 помилок
  ГОТОВО     етап 7: три висновки
  ГОТОВО     етап 7: відповідь на питання
  ГОТОВО     декларацію заповнено
  ГОТОВО     опис перевірки ≥ 120 символів
  ГОТОВО     підтвердження авторства

Робота варіанта 14 готова до захисту.
Недосіка Назарій, ШІ-32


---
### Перед здачею

Виконайте **Kernel → Restart & Run All**. Ноутбук має пройти згори донизу
без жодної помилки — це перше, що перевіряється на захисті.